# gold_visao_360

Tabelas da **Visão 360** da UC (Tela 3, Camila): o que o `gold_base` não traz e o `GET /unidades/{id}` precisa. Cadastro, medidor, GD e faturas vêm da `dim_uc` e do `fato_uc_mes`. Este notebook não os repete (regra 3: uma métrica, um lugar).

- **Entrada:** `gold.fato_uc_mes` e `gold.dim_uc` (gold_base), `silver.tb_ocorrencias_rede` e `silver.tb_atendimentos` (silver_rede_atendimento).
- **Saída:** `gold.uc_consumo_comparativo`, `gold.uc_interrupcoes` e `gold.uc_atendimentos`.
- **Depende de:** `gold_base` e `silver_rede_atendimento`. Roda em paralelo com as outras Gold.
- Segue o *Contrato de dados Gold → API* (seção 3.10) e as decisões do `exploracao_gold_visao_360`. Idempotente: grava em overwrite e pode ser reexecutado.

| Tabela | Grão (1 linha por) | Conteúdo |
|---|---|---|
| `uc_consumo_comparativo` | UC × mês | `consumo_kwh` (consumo válido), `consumo_kwh_ano_anterior`, `variacao_pct` (em %), `tipo_leitura` e `flag_conta_alta` |
| `uc_interrupcoes` | ocorrência | colunas da ocorrência + `duracao_efetiva_min` e as flags `flag_sem_interrupcao`, `flag_antes_ligacao`, `flag_dia_critico`, `flag_duracao_teto` e `flag_sobreposicao` |
| `uc_atendimentos` | chamado | colunas do chamado + `tempo_atendimento_h` |

**Regras**
- **Consumo:** usa o `consumo_kwh_valido` do `fato_uc_mes`, o mesmo do dashboard. As leituras negativas não recuperadas ficam nulas, nunca aparecem como consumo.
- **Ano anterior:** self-join por `mes_referencia − 12 meses`, sem `lag`, para não depender de a UC ter todos os meses. Sem ano anterior, fica nulo, nunca 0.
- **Variação:** em percentual (38,26 = alta de 38,26%), como no contrato da API. Nula quando falta o ano anterior ou quando algum dos dois meses tem consumo ≤ 0.
- **Conta alta:** consumo mais de **20%** acima do mesmo mês do ano anterior. É o p95 da variação e marca cerca de 5,4% dos meses. O `tipo_leitura` vai junto: uma leitura ESTIMADA pode disparar a flag sem que o consumo real tenha subido.
- **Interrupções:** todas as ocorrências da UC ficam na tabela, com as mesmas flags do DEC/FEC. A API decide o que mostrar: oscilação (`flag_sem_interrupcao`) não é falta de energia, e `flag_antes_ligacao` marca ocorrências de antes de a UC existir.

In [0]:
%run ../bronze-to-silver/funcoes_comuns

In [0]:
dbutils.widgets.text("catalog", "workspace")
CATALOGO = dbutils.widgets.get("catalog")

In [0]:
# ---------- Regras de negócio: Visão 360 ----------
LIMIAR_CONTA_ALTA = 20       # consumo > +20% vs mesmo mês do ano anterior (p95 da variação; exploracao_gold_visao_360)
MESES_ANO_ANTERIOR = 12

# Colunas de cada tabela, na ordem do contrato Gold → API (seção 3.10): negócio → flags
COLUNAS_COMPARATIVO = ["id_uc", "mes_referencia", "consumo_kwh", "consumo_kwh_ano_anterior",
                       "variacao_pct", "tipo_leitura", "flag_conta_alta"]
COLUNAS_INTERRUPCOES = ["id_ocorrencia", "id_uc", "alimentador", "tipo", "causa",
                        "data_abertura", "data_encerramento", "duracao_min", "duracao_efetiva_min",
                        "flag_sem_interrupcao", "flag_antes_ligacao", "flag_dia_critico",
                        "flag_duracao_teto", "flag_sobreposicao"]
COLUNAS_ATENDIMENTOS = ["protocolo", "id_uc", "canal", "assunto", "subassunto",
                        "data_abertura", "data_fechamento", "nota_satisfacao", "tempo_atendimento_h"]

In [0]:
gold = lambda t: spark.table(f"{CATALOGO}.gold.{t}")
silver = lambda t: spark.table(f"{CATALOGO}.silver.tb_{t}")   # as tabelas escolhem as colunas uma a uma

# Consumo válido: o mesmo do dashboard (sem as leituras negativas não recuperadas)
fato = gold("fato_uc_mes").select("id_uc", "mes_referencia",
                                  F.col("consumo_kwh_valido").alias("consumo_kwh"), "tipo_leitura")
ucs = gold("dim_uc").select("id_uc")             # referência para a integridade
oc, at = silver("ocorrencias_rede"), silver("atendimentos")

N_FATO, N_OC, N_AT = fato.count(), oc.count(), at.count()   # para a reconciliação no fim
print(f"fato_uc_mes: {N_FATO:,} | ocorrências: {N_OC:,} | atendimentos: {N_AT:,}")

In [0]:
# uc_consumo_comparativo: 1 linha por UC × mês = consumo do mês + consumo do mesmo mês do ano anterior
mes_mais_12 = F.date_format(F.add_months(F.to_date("mes_referencia", "yyyy-MM"), MESES_ANO_ANTERIOR), "yyyy-MM")
anterior = fato.select("id_uc", mes_mais_12.alias("mes_referencia"),
                       F.col("consumo_kwh").alias("consumo_kwh_ano_anterior"))

atual_kwh, ant_kwh = F.col("consumo_kwh"), F.col("consumo_kwh_ano_anterior")
comparavel = (atual_kwh > 0) & (ant_kwh > 0)      # falso ou nulo sem ano anterior ou com consumo ≤ 0
# Variação em percentual (38,26 = alta de 38,26%), como no contrato da API
variacao = F.when(comparavel, F.round((atual_kwh - ant_kwh) / ant_kwh * 100, 2).cast("decimal(10,2)"))

uc_consumo_comparativo = (fato.join(anterior, ["id_uc", "mes_referencia"], "left")
    .withColumn("variacao_pct", variacao)
    .withColumn("flag_conta_alta", F.col("variacao_pct") > LIMIAR_CONTA_ALTA)   # nula quando a variação é nula
    .select(*COLUNAS_COMPARATIVO))

In [0]:
# uc_interrupcoes: 1 linha por ocorrência = a ocorrência da Silver, com as flags que a tela usa
uc_interrupcoes = oc.select(*COLUNAS_INTERRUPCOES)

In [0]:
# uc_atendimentos: 1 linha por chamado = o chamado da Silver + tempo de atendimento em horas
tempo_h = (F.unix_timestamp("data_fechamento") - F.unix_timestamp("data_abertura")) / 3600
uc_atendimentos = (at.withColumn("tempo_atendimento_h", F.round(tempo_h, 2).cast("decimal(10,2)"))
                     .select(*COLUNAS_ATENDIMENTOS))

In [0]:
# Checagens: falham ANTES de gravar
TABELAS = {"uc_consumo_comparativo": (uc_consumo_comparativo, ["id_uc", "mes_referencia"], N_FATO),
           "uc_interrupcoes": (uc_interrupcoes, ["id_ocorrencia"], N_OC),
           "uc_atendimentos": (uc_atendimentos, ["protocolo"], N_AT)}

for nome, (df, chave, n_origem) in TABELAS.items():
    checar_unicidade(df, chave, nome)
    checar_sem_nulos(df, list(dict.fromkeys([*chave, "id_uc"])), nome)
    orfas = df.join(ucs, "id_uc", "left_anti").count()
    assert orfas == 0, f"{nome}: {orfas:,} linhas com UC fora da dim_uc"
    n = df.count()
    assert n == n_origem, f"{nome}: {n:,} linhas na Gold, {n_origem:,} na origem"

checar_sem_nulos(uc_interrupcoes, ["data_abertura", "data_encerramento", "duracao_min"], "uc_interrupcoes")
checar_sem_nulos(uc_atendimentos, ["data_abertura", "data_fechamento", "tempo_atendimento_h"], "uc_atendimentos")

negativo = uc_atendimentos.filter(F.col("tempo_atendimento_h") < 0).count()
assert negativo == 0, f"{negativo:,} chamados com tempo de atendimento negativo"

# Conferência (não para o notebook): a taxa de conta alta deve ficar perto dos 5,4% da exploração
r = uc_consumo_comparativo.agg(F.count("variacao_pct").alias("comparaveis"),
            F.sum(F.col("flag_conta_alta").cast("int")).alias("conta_alta")).first()
print(f"comparáveis: {r.comparaveis:,} | conta alta: {r.conta_alta:,} ({r.conta_alta / r.comparaveis:.1%})")

In [0]:
# Gravar (overwrite: reexecutável)
for nome, (df, _, _) in TABELAS.items():
    (df.withColumn("_atualizado_em", F.current_timestamp())
       .write.format("delta").mode("overwrite").option("overwriteSchema", "true")
       .saveAsTable(f"{CATALOGO}.gold.{nome}"))
    print(f"gold.{nome}: {spark.table(f'{CATALOGO}.gold.{nome}').count():,} linhas")